# Pig Posture Recognition – V3 (Domain-Shift Focus)

**Problem:** Val F1 ~0.90, aber Kaggle nur ~0.60 → massiver Domain-Shift zwischen Train- und Test-Kameras.

**Analyse-Erkenntnisse:**
- Test hat NUR 3 Kameras: `pen1_tur_cam1` (36%), `pen2_orb_cam2` (24%), `pen2_tur_cam2` (40%)
- Alle 3 existieren im Training, aber mit winzigen Mengen (120–200 Instanzen)
- Aufloesungs-Shift: Test = 1280x720, Training = 1920x1080 / 1280x720 / 2688x1520 gemischt
- Zeitlicher Shift: Train = Jan/Feb 2025, Test = Sep 2025 → komplett andere Schweine
- T2-Bonus fuegt genau diese 3 Test-Kameras hinzu (+516 Instanzen)
- Klassen-Imbalance: Sitting nur 3%, Standing 42% → Macro-F1 bestraft Sitting-Fehler stark

**Loesungen:**
1. **DINOv2 Backbone** – Selbst-ueberwacht auf 142M Bildern, beste Domain-Generalisierung
2. **Camera-Leave-Out (nur Test-Kameras)** – 3 Folds statt 8, simuliert exakt den Kaggle-Test
3. **Differential Learning Rate** – Backbone langsam, Head schnell fine-tunen
4. **Label-aware Horizontal Flip** – Links/Rechts korrekt getauscht
5. **Pseudo-Labeling Support** – Iteratives Self-Training auf Test-Domain

## Configuration

In [1]:
TAG = "T2"   # "T1" oder "T2"

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

_candidates = [
    'multiview_pig_posture_recognition',
    './multiview_pig_posture_recognition',
    '/datasets/multi-view-pig-posture-recognition',
    '/multi-view-pig-posture-recognition',
]
DATA_ROOT = None
for _p in _candidates:
    if os.path.isdir(_p):
        DATA_ROOT = _p
        break
assert DATA_ROOT is not None, f'Datenverzeichnis nicht gefunden!'
print(f'DATA_ROOT = {os.path.abspath(DATA_ROOT)}')

if TAG == "T1":
    CSV_PATH = f"{DATA_ROOT}/train1.csv"
    IMG_DIR  = f"{DATA_ROOT}/train1_images"
else:
    CSV_PATH = f"{DATA_ROOT}/train2.csv"
    IMG_DIR  = f"{DATA_ROOT}/train2_images"

OUTPUT_DIR = f"runs/v3_{TAG.lower()}"

# ─── Modell ───
MODEL_NAME = "vit_base_patch14_dinov2.lvd142m"
MODEL_FALLBACKS = [
    "vit_base_patch14_reg4_dinov2.lvd142m",
    "convnextv2_base.fcmae_ft_in22k_in1k_384",
    "convnext_base",
]
# DINOv2 patch14: muss durch 14 teilbar sein!
# Default waere 518 (37*14), aber 392 (28*14) spart VRAM
# Wird via img_size= an timm.create_model uebergeben
IMG_SIZE          = 392
BATCH_SIZE        = 32           # 2 GPUs → 8 pro GPU
EPOCHS            = 20
WARMUP_EPOCHS     = 3            # Linear Warmup fuer DINOv2 Fine-Tuning
LR                = 1e-4
LR_BACKBONE_MULT  = 0.1         # Backbone bekommt LR * 0.1
LABEL_SMOOTH      = 0.05
MIXUP_ALPHA       = 0.1         # Niedrig: MixUp kann bei Camera-Leave-Out kontraproduktiv sein
PAD_RATIO         = 0.1         # Weniger Hintergrund = weniger kameraspezifische Features
NUM_WORKERS       = 8
SEED              = 42
NUM_CLASSES       = 5

CLASS_NAMES = ["Lateral_lying_left", "Lateral_lying_right",
               "Sitting", "Standing", "Sternal_lying"]

# ─── Test-Kameras (aus Analyse) ───
# NUR diese 3 Kameras kommen im Test vor:
#   pen1_tur_cam1  → 36.4% des Tests (4264 Instanzen)
#   pen2_orb_cam2  → 24.0% des Tests (2805 Instanzen)
#   pen2_tur_cam2  → 39.6% des Tests (4639 Instanzen)
# Im Training existieren sie nur mit 200/120/196 Instanzen!
TEST_CAMERAS = ["pen1_tur_cam1", "pen2_orb_cam2", "pen2_tur_cam2"]

# ─── Validation-Strategie ───
# "camera" = Camera-Leave-Out NUR auf Test-Kameras (3 Folds, simuliert Kaggle exakt)
# "image"  = StratifiedGroupKFold nach image_id (5 Folds, kein CLO)
VALIDATION_STRATEGY = "camera"

# ─── Pseudo-Labeling (Phase 2) ───
USE_PSEUDO_LABELS = False
PSEUDO_CSV        = None   # z.B. "pseudo_labels_t1.csv" vom Inference-Notebook

# ─── Fine-Tuning ───
PRETRAINED_CKPT = None

print(f"Tag: {TAG}  |  Model: {MODEL_NAME}  |  Res: {IMG_SIZE}px")
print(f"Validation: {VALIDATION_STRATEGY}  |  BS: {BATCH_SIZE}  |  Epochs: {EPOCHS}")
print(f"LR: {LR} (Backbone x{LR_BACKBONE_MULT})  |  Warmup: {WARMUP_EPOCHS} Epochen")
print(f"MixUp: {MIXUP_ALPHA}  |  PAD_RATIO: {PAD_RATIO}  |  Output: {OUTPUT_DIR}")
print(f"Test-Kameras: {TEST_CAMERAS}")

DATA_ROOT = /datasets/multi-view-pig-posture-recognition
Tag: T2  |  Model: vit_base_patch14_dinov2.lvd142m  |  Res: 392px
Validation: camera  |  BS: 32  |  Epochs: 20
LR: 0.0001 (Backbone x0.1)  |  Warmup: 3 Epochen
MixUp: 0.1  |  PAD_RATIO: 0.1  |  Output: runs/v3_t2
Test-Kameras: ['pen1_tur_cam1', 'pen2_orb_cam2', 'pen2_tur_cam2']


## Imports

In [2]:
import os, ast, random, re
import numpy as np
import pandas as pd
from PIL import Image, ImageFilter
from io import BytesIO
from tqdm.notebook import tqdm
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as T
import torchvision.transforms.functional as TFn
import timm

from sklearn.model_selection import LeaveOneGroupOut, StratifiedGroupKFold
from sklearn.metrics import f1_score, classification_report

import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPUs: {torch.cuda.device_count()}")

<jemalloc>: Unsupported system page size


Device: cuda
GPU: Tesla V100-SXM2-32GB
GPUs: 2


In [3]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Daten laden & Kamera-Analyse

In [4]:
df = pd.read_csv(CSV_PATH)

def extract_camera(image_id):
    m = re.match(r'(pen\d+_\w+_cam\d+)', image_id)
    return m.group(1) if m else 'unknown'

df['camera'] = df['image_id'].apply(extract_camera)
df['img_dir'] = IMG_DIR

# ─── Optional: Pseudo-Labels ───
if USE_PSEUDO_LABELS and PSEUDO_CSV and os.path.exists(PSEUDO_CSV):
    pseudo_df = pd.read_csv(PSEUDO_CSV)
    # Nur die Spalten behalten, die wir brauchen (confidence etc. rausfiltern)
    keep_cols = [c for c in pseudo_df.columns if c in ['row_id', 'image_id', 'width', 'height', 'bbox', 'class_id']]
    pseudo_df = pseudo_df[keep_cols]
    pseudo_df['camera'] = pseudo_df['image_id'].apply(extract_camera)
    pseudo_df['img_dir'] = os.path.join(DATA_ROOT, 'test_images')
    df = pd.concat([df, pseudo_df], ignore_index=True)
    print(f"Pseudo-Labels: {len(pseudo_df)} hinzugefuegt -> Gesamt: {len(df)}")

print(f"Instanzen: {len(df)}  |  Bilder: {df['image_id'].nunique()}")
print(f"\nKameras ({df['camera'].nunique()}):")
for cam in sorted(df['camera'].unique()):
    cnt = (df['camera'] == cam).sum()
    bar = '█' * int(40 * cnt / len(df))
    print(f"  {cam:<25} {bar:<40} {cnt:>5} ({100*cnt/len(df):.1f}%)")

print(f"\nKlassen:")
for c in range(NUM_CLASSES):
    cnt = (df['class_id'] == c).sum()
    print(f"  {c} - {CLASS_NAMES[c]:<22} {cnt:>5} ({100*cnt/len(df):.1f}%)")

Instanzen: 23450  |  Bilder: 3150

Kameras (8):
  pen1_orb_cam1             █                                          722 (3.1%)
  pen1_orb_cam2             ██                                        1488 (6.3%)
  pen1_tur_cam1                                                        200 (0.9%)
  pen1_tur_cam2             █████████████                             8194 (34.9%)
  pen2_orb_cam1             ████                                      2817 (12.0%)
  pen2_orb_cam2                                                        120 (0.5%)
  pen2_tur_cam1             ████████████████                          9713 (41.4%)
  pen2_tur_cam2                                                        196 (0.8%)

Klassen:
  0 - Lateral_lying_left      3083 (13.1%)
  1 - Lateral_lying_right     3435 (14.6%)
  2 - Sitting                  695 (3.0%)
  3 - Standing                9928 (42.3%)
  4 - Sternal_lying           6309 (26.9%)


## Dataset mit Label-aware Flip

Horizontal Flip tauscht Label 0 (left) ↔ 1 (right). Wird im Dataset gemacht, nicht in der Transform-Pipeline.

In [5]:
class PigPostureDataset(Dataset):
    def __init__(self, df, transform=None, pad_ratio=0.25,
                 is_train=False, hflip_prob=0.5):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.pad_ratio = pad_ratio
        self.is_train = is_train
        self.hflip_prob = hflip_prob

    def __len__(self):
        return len(self.df)

    def _crop(self, img, bbox):
        W, H = img.size
        x, y, w, h = [float(v) for v in ast.literal_eval(bbox)]
        px, py = w * self.pad_ratio, h * self.pad_ratio
        x1 = max(0, int(x - px));  y1 = max(0, int(y - py))
        x2 = min(W, int(x+w+px));  y2 = min(H, int(y+h+py))
        return img.crop((x1, y1, x2, y2))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(row['img_dir'], row['image_id'])).convert('RGB')
        crop = self._crop(img, row['bbox'])
        label = int(row['class_id'])

        if self.is_train and random.random() < self.hflip_prob:
            crop = TFn.hflip(crop)
            if label == 0:
                label = 1
            elif label == 1:
                label = 0

        if self.transform:
            crop = self.transform(crop)
        return crop, label

## Augmentierungen

**Analyse-basierte Anpassungen:**
- CameraSimTransform simuliert Aufloesungs-Shift (Test-Crops sind ~224px, Train-Crops ~300-480px)
- Downscale-Augmentation: Crop verkleinern und wieder hochskalieren → simuliert kleinere Test-Bboxen
- Keine zu aggressiven Augmentierungen – DINOv2 hat bereits robuste Features

In [6]:
class CameraSimTransform:
    """Simuliert Kamera-Domain-Shift.

    Aus der Analyse:
    - Train-Crops: ~300-480px, Test-Crops: ~224px (deutlich kleiner)
    - Test-Aufloesung: 1280x720 (einheitlich), Train: gemischt (1920x1080, 1280x720, 2688x1520)
    - Downscale simuliert den Effekt kleinerer Bboxen in niedrigerer Aufloesung
    """
    def __init__(self, p=0.4):
        self.p = p

    def __call__(self, img):
        if random.random() > self.p:
            return img

        # Downscale + Upscale: simuliert kleinere Crops / niedrigere Aufloesung
        # Aggressiver als vorher (0.3-0.7 statt 0.4-0.8), weil Test-Crops ~50% kleiner sind
        if random.random() < 0.4:
            w, h = img.size
            scale = random.uniform(0.3, 0.7)
            small = img.resize((max(16, int(w*scale)), max(16, int(h*scale))), Image.BILINEAR)
            img = small.resize((w, h), Image.BILINEAR)

        # JPEG-Kompression
        if random.random() < 0.2:
            quality = random.randint(25, 65)
            buffer = BytesIO()
            img.save(buffer, format='JPEG', quality=quality)
            buffer.seek(0)
            img = Image.open(buffer).convert('RGB')

        # Gauss-Rauschen
        if random.random() < 0.15:
            arr = np.array(img, dtype=np.float32)
            noise = np.random.normal(0, random.uniform(5, 15), arr.shape)
            arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
            img = Image.fromarray(arr)

        return img


class PerspectiveJitter:
    def __init__(self, distortion_scale=0.10, p=0.25):
        self.distortion_scale = distortion_scale
        self.p = p

    def __call__(self, img):
        if random.random() < self.p:
            d = self.distortion_scale
            w, h = img.size
            return TFn.perspective(
                img,
                startpoints=[[0,0],[w,0],[w,h],[0,h]],
                endpoints=[
                    [int(random.uniform(0, w*d)), int(random.uniform(0, h*d))],
                    [int(w - random.uniform(0, w*d)), int(random.uniform(0, h*d))],
                    [int(w - random.uniform(0, w*d)), int(h - random.uniform(0, h*d))],
                    [int(random.uniform(0, w*d)), int(h - random.uniform(0, h*d))],
                ],
                fill=0
            )
        return img


def get_train_transform(size=IMG_SIZE):
    return T.Compose([
        # Camera-Simulation (Aufloesungs-Shift + Kompression)
        CameraSimTransform(p=0.4),
        PerspectiveJitter(distortion_scale=0.10, p=0.25),
        T.Resize((size, size), interpolation=T.InterpolationMode.BICUBIC),
        # KEIN HorizontalFlip – wird im Dataset mit Label-Swap gemacht
        T.RandomRotation(degrees=12),
        T.RandomAffine(degrees=0, scale=(0.85, 1.15), translate=(0.05, 0.05)),
        # Farbe (abgeschwaecht – DINOv2 ist robust gegen Farbvariationen)
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
        T.RandomGrayscale(p=0.08),
        # Tensor
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        T.RandomErasing(p=0.20, scale=(0.02, 0.12)),
    ])

def get_val_transform(size=IMG_SIZE):
    return T.Compose([
        T.Resize((size, size), interpolation=T.InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

print("Augmentierungen definiert (angepasst an Analyse-Erkenntnisse).")

Augmentierungen definiert (angepasst an Analyse-Erkenntnisse).


## Training Helpers

**Differential Learning Rate:** DINOv2-Backbone bekommt eine 10x niedrigere LR als der Klassifikations-Head.
So bleiben die starken vortrainierten Features erhalten.

In [7]:
def mixup_data(x, y, alpha=0.2):
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_loss(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


def build_optimizer(model, lr, lr_backbone_mult, weight_decay=1e-2):
    """Differential LR: Backbone langsamer, Head schneller."""
    head_params = []
    backbone_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        # DataParallel wraps mit 'module.'
        clean_name = name.replace('module.', '')
        if clean_name.startswith('head') or clean_name.startswith('fc') or clean_name.startswith('classifier'):
            head_params.append(param)
        else:
            backbone_params.append(param)

    print(f"  Optimizer: {len(backbone_params)} backbone params (LR={lr*lr_backbone_mult:.1e}), "
          f"{len(head_params)} head params (LR={lr:.1e})")

    return optim.AdamW([
        {'params': backbone_params, 'lr': lr * lr_backbone_mult},
        {'params': head_params,     'lr': lr},
    ], weight_decay=weight_decay)


def train_one_epoch(model, loader, optimizer, scaler, criterion):
    model.train()
    optimizer.zero_grad()
    loss_sum, n_samples = 0.0, 0
    preds_all, labels_all = [], []

    for imgs, labels in tqdm(loader, desc="  Train", leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        imgs_mix, y_a, y_b, lam = mixup_data(imgs, labels, alpha=MIXUP_ALPHA)

        with autocast():
            logits = model(imgs_mix)
            loss = mixup_loss(criterion, logits, y_a, y_b, lam)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

        loss_sum += loss.item() * imgs.size(0)
        n_samples += imgs.size(0)
        preds_all.extend(logits.argmax(1).cpu().numpy())
        labels_all.extend(y_a.cpu().numpy())

    return (loss_sum / max(n_samples, 1), f1_score(labels_all, preds_all, average="macro", zero_division=0)) if n_samples > 0 else (0.0, 0.0)


@torch.no_grad()
def validate_epoch(model, loader, criterion):
    model.eval()
    loss_sum, n_samples = 0.0, 0
    preds_all, labels_all = [], []

    for imgs, labels in tqdm(loader, desc="  Val  ", leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast():
            logits = model(imgs)
            loss = criterion(logits, labels)
        loss_sum += loss.item() * imgs.size(0)
        n_samples += imgs.size(0)
        preds_all.extend(logits.argmax(1).cpu().numpy())
        labels_all.extend(labels.cpu().numpy())

    return (loss_sum / max(n_samples, 1),
            f1_score(labels_all, preds_all, average="macro", zero_division=0),
            preds_all, labels_all)

## Camera-Leave-Out Training

**Automatische Erkennung:**
- **T2** (Test-Kameras im Datensatz): CLO auf genau die 3 Test-Kameras -> 3 Folds
- **T1** (Test-Kameras NICHT im Datensatz): CLO auf alle vorhandenen Kameras -> 5-8 Folds

**T1-Modus:** Die 3 Test-Kameras (pen1_tur_cam1, pen2_orb_cam2, pen2_tur_cam2) existieren
NUR in T2. Bei T1-Training muessen wir auf alle Kameras ausweichen.
Die Val-F1 Werte sind dann weniger direkt vergleichbar mit Kaggle, aber immernoch besser als image-level Split.

In [8]:
df_train = df.copy()

# ─── Folds aufbauen ───
if VALIDATION_STRATEGY == "camera":
    available_cams = set(df_train['camera'].unique())
    test_cams_in_data = sorted([c for c in TEST_CAMERAS if c in available_cams])

    if len(test_cams_in_data) > 0:
        # T2-Modus: Test-Kameras sind im Datensatz → CLO direkt darauf
        print(f"Test-Kameras im Datensatz: {test_cams_in_data}")
        cams_for_clo = test_cams_in_data
    else:
        # T1-Modus: Test-Kameras NICHT im Datensatz
        # → CLO auf ALLE Kameras (jede koennte dem Test aehneln)
        print(f"KEINE Test-Kameras im Datensatz (T1-Modus)")
        print(f"  → CLO auf alle {len(available_cams)} Kameras")
        cams_for_clo = sorted(available_cams)

    splits = []
    fold_cameras = []
    for cam in cams_for_clo:
        val_mask = df_train['camera'] == cam
        val_idx = df_train.index[val_mask].values
        train_idx = df_train.index[~val_mask].values
        if len(val_idx) > 0:  # Nur Folds mit Val-Daten
            splits.append((train_idx, val_idx))
            fold_cameras.append(cam)

    n_folds = len(splits)
    print(f"\nCamera-Leave-Out: {n_folds} Folds")
    for i, cam in enumerate(fold_cameras):
        cnt = (df_train['camera'] == cam).sum()
        train_size = len(df_train) - cnt
        print(f"  Fold {i+1}: Val = {cam} ({cnt} Instanzen) | Train = {train_size}")
else:
    n_folds = 5
    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    splits = list(sgkf.split(X=df_train, y=df_train['class_id'], groups=df_train['image_id']))
    fold_cameras = [None] * n_folds
    print(f"StratifiedGroupKFold: {n_folds} Folds (nach image_id)")

fold_results = []

for fold_idx, (train_idx, val_idx) in enumerate(splits):
    print(f"\n{'='*60}")
    print(f"  FOLD {fold_idx + 1} / {n_folds}")
    print(f"{'='*60}")

    fold_train = df_train.iloc[train_idx].reset_index(drop=True)
    fold_val   = df_train.iloc[val_idx].reset_index(drop=True)

    if VALIDATION_STRATEGY == "camera":
        val_cam = fold_cameras[fold_idx]
        train_cams = sorted(fold_train['camera'].unique())
        val_classes = sorted(fold_val['class_id'].unique())
        print(f"  Val-Kamera: {val_cam}")
        print(f"  Train-Kameras: {train_cams}")
        print(f"  Val-Klassen: {len(val_classes)}/5", end="")
        missing = [CLASS_NAMES[c] for c in range(NUM_CLASSES) if c not in val_classes]
        if missing:
            print(f"  (FEHLT: {', '.join(missing)})", end="")
        print()

    print(f"  Train: {len(fold_train)} | Val: {len(fold_val)}")

    # ─── Skip wenn Checkpoint existiert ───
    ckpt_path = os.path.join(OUTPUT_DIR, f"best_model_fold_{fold_idx+1}.pth")
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location='cpu')
        old_f1 = ckpt.get('val_f1', 0)
        old_cam = ckpt.get('val_camera', '?')
        print(f"  Checkpoint existiert (val_f1={old_f1:.4f}, cam={old_cam}), ueberspringe...")
        fold_results.append(old_f1)
        continue

    train_ds = PigPostureDataset(fold_train, transform=get_train_transform(),
                                  pad_ratio=PAD_RATIO, is_train=True, hflip_prob=0.5)
    val_ds   = PigPostureDataset(fold_val,   transform=get_val_transform(),
                                  pad_ratio=PAD_RATIO, is_train=False)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

    # ─── Modell ───
    model = None
    used_name = MODEL_NAME
    for name in [MODEL_NAME] + MODEL_FALLBACKS:
        try:
            model = timm.create_model(name, pretrained=True, num_classes=NUM_CLASSES, img_size=IMG_SIZE)
            used_name = name
            break
        except Exception:
            print(f"  '{name}' nicht verfuegbar, naechstes...")
            continue
    assert model is not None, "Kein Modell konnte geladen werden!"

    if PRETRAINED_CKPT and os.path.exists(PRETRAINED_CKPT):
        ckpt = torch.load(PRETRAINED_CKPT, map_location='cpu')
        model.load_state_dict(ckpt['model'], strict=False)
        print(f"  Checkpoint geladen: {PRETRAINED_CKPT}")

    model = model.to(DEVICE)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)

    params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  Modell: {used_name} ({params:.1f}M)")

    # ─── Loss mit Klassen-Gewichtung ───
    counts = Counter(fold_train['class_id'].tolist())
    weights = torch.tensor(
        [len(fold_train) / (NUM_CLASSES * max(counts.get(c, 1), 1))
         for c in range(NUM_CLASSES)], dtype=torch.float32
    ).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=LABEL_SMOOTH)
    print(f"  Klassen-Gewichte: {[f'{w:.2f}' for w in weights.cpu().tolist()]}")

    # ─── Optimizer mit Differential LR + Warmup ───
    optimizer = build_optimizer(model, LR, LR_BACKBONE_MULT)

    warmup = LinearLR(optimizer, start_factor=0.01, total_iters=WARMUP_EPOCHS)
    cosine = CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP_EPOCHS, eta_min=1e-7)
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS])
    print(f"  Scheduler: {WARMUP_EPOCHS}ep Warmup -> Cosine Decay")

    scaler = GradScaler()

    best_val_f1 = 0.0
    patience_counter = 0
    PATIENCE = 7

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_f1 = train_one_epoch(model, train_loader, optimizer, scaler, criterion)
        val_loss, val_f1, val_preds, val_labels = validate_epoch(model, val_loader, criterion)
        scheduler.step()

        improved = val_f1 > best_val_f1
        mark = "★" if improved else " "
        phase = "warmup" if epoch <= WARMUP_EPOCHS else "cosine"
        print(f"  {mark} Epoch {epoch:02d}/{EPOCHS} [{phase}] | "
              f"Train L={train_loss:.4f} F1={train_f1:.4f} | "
              f"Val L={val_loss:.4f} F1={val_f1:.4f}"
              f"{' <- BEST' if improved else ''}")

        if improved:
            best_val_f1 = val_f1
            patience_counter = 0
            state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            torch.save({
                "epoch": epoch, "model": state,
                "val_f1": val_f1, "model_name": used_name,
                "tag": TAG, "img_size": IMG_SIZE, "pad_ratio": PAD_RATIO,
                "val_camera": fold_cameras[fold_idx],
            }, ckpt_path)
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"  Early Stopping nach {PATIENCE} Epochen ohne Verbesserung")
                break

    # Per-Klasse Report fuer diesen Fold
    print(f"\n  Klassifikation (Fold {fold_idx+1}, Val={fold_cameras[fold_idx]}):")
    for c in range(NUM_CLASSES):
        mask = np.array(val_labels) == c
        if mask.sum() > 0:
            correct = (np.array(val_preds)[mask] == c).sum()
            print(f"    {CLASS_NAMES[c]:<22} {correct}/{mask.sum()} ({100*correct/max(mask.sum(),1):.0f}%)")
        else:
            print(f"    {CLASS_NAMES[c]:<22} nicht in Validation")

    fold_results.append(best_val_f1)
    print(f"\n  Fold {fold_idx+1} fertig! Best Val F1: {best_val_f1:.4f}")

    del model, optimizer, scheduler, scaler
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print(f"  TRAINING ABGESCHLOSSEN")
print(f"{'='*60}")
print(f"F1 pro Fold:")
for i, (f1, cam) in enumerate(zip(fold_results, fold_cameras)):
    cam_label = f" ({cam})" if cam else ""
    print(f"  Fold {i+1}{cam_label}: {f1:.4f}")
mean_f1 = sum(fold_results) / len(fold_results)
std_f1  = np.std(fold_results)
print(f"\nDurchschnitt: {mean_f1:.4f} (+/- {std_f1:.4f})")
print(f"\nDieser Wert sollte naeher am Kaggle-Score liegen als die 0.90 vorher!")

Test-Kameras im Datensatz: ['pen1_tur_cam1', 'pen2_orb_cam2', 'pen2_tur_cam2']

Camera-Leave-Out: 3 Folds
  Fold 1: Val = pen1_tur_cam1 (200 Instanzen) | Train = 23250
  Fold 2: Val = pen2_orb_cam2 (120 Instanzen) | Train = 23330
  Fold 3: Val = pen2_tur_cam2 (196 Instanzen) | Train = 23254

  FOLD 1 / 3
  Val-Kamera: pen1_tur_cam1
  Train-Kameras: ['pen1_orb_cam1', 'pen1_orb_cam2', 'pen1_tur_cam2', 'pen2_orb_cam1', 'pen2_orb_cam2', 'pen2_tur_cam1', 'pen2_tur_cam2']
  Val-Klassen: 5/5
  Train: 23250 | Val: 200
  Modell: vit_base_patch14_dinov2.lvd142m (86.1M)
  Klassen-Gewichte: ['1.51', '1.36', '6.72', '0.48', '0.74']
  Optimizer: 174 backbone params (LR=1.0e-05), 2 head params (LR=1.0e-04)
  Scheduler: 3ep Warmup -> Cosine Decay


  Train:   0%|          | 0/726 [00:00<?, ?it/s]

  Val  :   0%|          | 0/7 [00:00<?, ?it/s]

  ★ Epoch 01/20 [warmup] | Train L=1.6665 F1=0.2325 | Val L=1.6404 F1=0.2609 <- BEST


  Train:   0%|          | 0/726 [00:00<?, ?it/s]

...

  Val  :   0%|          | 0/7 [00:00<?, ?it/s]

    Epoch 18/20 [cosine] | Train L=0.5629 F1=0.5986 | Val L=0.9287 F1=0.8237


  Train:   0%|          | 0/726 [00:00<?, ?it/s]

  Val  :   0%|          | 0/7 [00:00<?, ?it/s]

    Epoch 19/20 [cosine] | Train L=0.5590 F1=0.6003 | Val L=0.9253 F1=0.8051
  Early Stopping nach 7 Epochen ohne Verbesserung

  Klassifikation (Fold 3, Val=pen2_tur_cam2):
    Lateral_lying_left     11/15 (73%)
    Lateral_lying_right    15/23 (65%)
    Sitting                8/11 (73%)
    Standing               111/112 (99%)
    Sternal_lying          27/35 (77%)

  Fold 3 fertig! Best Val F1: 0.8617

  TRAINING ABGESCHLOSSEN
F1 pro Fold:
  Fold 1 (pen1_tur_cam1): 0.8931
  Fold 2 (pen2_orb_cam2): 0.7899
  Fold 3 (pen2_tur_cam2): 0.8617

Durchschnitt: 0.8482 (+/- 0.0432)

Dieser Wert sollte naeher am Kaggle-Score liegen als die 0.90 vorher!


## Klassifikations-Report (letzter Fold)

In [9]:
if 'val_labels' in dir() and 'val_preds' in dir():
    print(classification_report(val_labels, val_preds, target_names=CLASS_NAMES))

                     precision    recall  f1-score   support

 Lateral_lying_left       1.00      0.73      0.85        15
Lateral_lying_right       0.62      0.65      0.64        23
            Sitting       0.89      0.73      0.80        11
           Standing       0.95      0.99      0.97       112
      Sternal_lying       0.77      0.77      0.77        35

           accuracy                           0.88       196
          macro avg       0.85      0.78      0.81       196
       weighted avg       0.88      0.88      0.88       196

